# Advanced Tutorial: Sending Data to Python Generators

## Problems, guided experiments, and complete solutions

This notebook develops the two-way generator protocol step by step. Instead of treating `yield` only as a way to produce values, we will use it as a suspension point that can also receive values through `send()`.

The notebook follows a tutorial rhythm:

1. build a precise mental model,
2. predict behavior before running code,
3. solve a focused problem,
4. study a complete solution,
5. verify it with assertions and edge cases,
6. extract a reusable best practice.

The exercises become progressively more demanding and culminate in a configurable streaming telemetry engine.


## Learning objectives

By the end, you should be able to:

- explain the generator lifecycle and the role of priming;
- distinguish the value yielded outward from the value sent inward;
- design command-driven coroutines with explicit message types;
- recover from injected exceptions with `throw()`;
- release resources reliably with `close()` and `finally`;
- retrieve generator return values from `StopIteration.value`;
- delegate a two-way protocol with `yield from`;
- build state machines, routers, and streaming analytics components;
- test coroutine behavior without depending on printed output.


In [1]:
from __future__ import annotations

from collections import deque
from contextlib import contextmanager
from dataclasses import dataclass
from enum import Enum, auto
from inspect import getgeneratorstate
from math import isclose, sqrt
from numbers import Real
from typing import Any, Callable, Generator, Iterable, Iterator, TypeVar


---

# 1. The paused-frame mental model

A generator object is a suspended execution frame. Each resume operation does two things:

1. it supplies a value to the currently suspended `yield` expression;
2. execution continues until the next `yield`, `return`, or uncaught exception.

The first resume is special because no `yield` expression is waiting yet. Therefore a just-created generator can receive only `None`.


## Guided example: trace both directions

Before running the next cell, predict the order of the printed messages and returned values.


In [2]:
def protocol_probe() -> Generator[str, str | None, str]:
    print('A: generator started')
    incoming = yield 'ready'
    print(f'B: received {incoming!r}')
    incoming = yield f'ack:{incoming}'
    print(f'C: received {incoming!r}')
    return 'finished'

probe = protocol_probe()
print('state 0:', getgeneratorstate(probe))
print('next ->', next(probe))
print('state 1:', getgeneratorstate(probe))
print("send('alpha') ->", probe.send('alpha'))
print('state 2:', getgeneratorstate(probe))

try:
    probe.send('omega')
except StopIteration as exc:
    print('return value:', exc.value)

print('state 3:', getgeneratorstate(probe))


state 0: GEN_CREATED
A: generator started
next -> ready
state 1: GEN_SUSPENDED
B: received 'alpha'
send('alpha') -> ack:alpha
state 2: GEN_SUSPENDED
C: received 'omega'
return value: finished
state 3: GEN_CLOSED


## Problem 1 — Build a lifecycle recorder

Write `record_lifecycle(factory, operations)`.

- `factory` is a zero-argument function that creates a generator.
- `operations` is a sequence containing `('next', None)` or `('send', value)`.
- Record the state immediately before and after every operation.
- Record yielded values.
- If the generator terminates, record its return value instead of allowing `StopIteration` to escape.

Return a list of dictionaries. The function must also handle invalid first sends by recording the exception type.


### Hint

Use `getgeneratorstate()` around a `try` / `except` block. The result of `next(g)` or `g.send(value)` is the value yielded outward.


### Solution 1


In [3]:
def record_lifecycle(
    factory: Callable[[], Generator[Any, Any, Any]],
    operations: Iterable[tuple[str, Any]],
) -> list[dict[str, Any]]:
    generator = factory()
    history: list[dict[str, Any]] = []

    for operation, payload in operations:
        entry: dict[str, Any] = {
            'operation': operation,
            'payload': payload,
            'state_before': getgeneratorstate(generator),
        }

        try:
            if operation == 'next':
                entry['yielded'] = next(generator)
            elif operation == 'send':
                entry['yielded'] = generator.send(payload)
            else:
                raise ValueError(f'unknown operation: {operation!r}')
        except StopIteration as exc:
            entry['returned'] = exc.value
        except Exception as exc:  # diagnostic recorder: preserve the error as data
            entry['exception'] = type(exc).__name__
            entry['message'] = str(exc)

        entry['state_after'] = getgeneratorstate(generator)
        history.append(entry)

    return history

history = record_lifecycle(
    protocol_probe,
    [('next', None), ('send', 'alpha'), ('send', 'omega')],
)

for row in history:
    print(row)


A: generator started
B: received 'alpha'
C: received 'omega'
{'operation': 'next', 'payload': None, 'state_before': 'GEN_CREATED', 'yielded': 'ready', 'state_after': 'GEN_SUSPENDED'}
{'operation': 'send', 'payload': 'alpha', 'state_before': 'GEN_SUSPENDED', 'yielded': 'ack:alpha', 'state_after': 'GEN_SUSPENDED'}
{'operation': 'send', 'payload': 'omega', 'state_before': 'GEN_SUSPENDED', 'returned': 'finished', 'state_after': 'GEN_CLOSED'}


In [4]:
assert history[0]['state_before'] == 'GEN_CREATED'
assert history[0]['yielded'] == 'ready'
assert history[1]['yielded'] == 'ack:alpha'
assert history[2]['returned'] == 'finished'
assert history[2]['state_after'] == 'GEN_CLOSED'

bad_start = record_lifecycle(protocol_probe, [('send', 'too early')])
assert bad_start[0]['exception'] == 'TypeError'
assert bad_start[0]['state_after'] == 'GEN_CREATED'


### Best practice

When learning or debugging a coroutine, inspect its lifecycle explicitly. Avoid reasoning only from printed output; record yielded values, termination values, states, and exceptions separately.


---

# 2. Priming safely

A coroutine that begins with `message = yield ...` must be advanced to its first suspension point before receiving a non-`None` value.

Manual priming is simple, but repeating `next(coroutine)` throughout an application is easy to forget. A small helper can make the protocol explicit.


## Problem 2 — Create a typed priming helper

Implement `prime(generator)` with these requirements:

- it accepts only a generator in `GEN_CREATED` state;
- it advances the generator exactly once;
- it returns both the generator and the first yielded value;
- it raises a clear error if the generator terminates during priming;
- it raises a clear error if the generator was already started.


### Solution 2


In [5]:
YieldT = TypeVar('YieldT')
SendT = TypeVar('SendT')
ReturnT = TypeVar('ReturnT')


def prime(
    generator: Generator[YieldT, SendT, ReturnT],
) -> tuple[Generator[YieldT, SendT, ReturnT], YieldT]:
    state = getgeneratorstate(generator)
    if state != 'GEN_CREATED':
        raise RuntimeError(f'generator must be newly created, not {state}')

    try:
        first_yield = next(generator)
    except StopIteration as exc:
        raise RuntimeError(
            f'generator terminated while being primed; return value={exc.value!r}'
        ) from exc

    return generator, first_yield


In [6]:
def receiver() -> Generator[str, str, None]:
    message = yield 'receiver-ready'
    while True:
        message = yield message.upper()

receiver_gen, greeting = prime(receiver())
assert greeting == 'receiver-ready'
assert receiver_gen.send('hello') == 'HELLO'
assert receiver_gen.send('coroutines') == 'COROUTINES'

try:
    prime(receiver_gen)
except RuntimeError as exc:
    print(exc)
finally:
    receiver_gen.close()


generator must be newly created, not GEN_SUSPENDED


### Mini-problem

Why does `generator.send(None)` work on a newly created generator?

Because sending `None` is defined to perform the same initial advance as `next(generator)`. A non-`None` value cannot be delivered because no suspended `yield` expression exists yet.


In [7]:
g = receiver()
assert g.send(None) == 'receiver-ready'
g.close()


### Best practice

Prime at one well-defined boundary: a factory, helper, decorator, or context manager. Do not scatter priming calls across business logic.


---

# 3. Structured commands instead of magic values

Using strings such as `'reset'` and `'stop'` can work in tiny demonstrations, but strings collide easily with real data and are hard to type-check. Explicit command objects make a coroutine protocol easier to understand and extend.


In [8]:
class TotalCommand(Enum):
    SNAPSHOT = auto()
    RESET = auto()
    STOP = auto()


@dataclass(frozen=True)
class TotalState:
    total: float
    count: int
    accepted: bool = True
    note: str = ''


## Problem 3 — Command-driven running total

Create `running_total()`.

Protocol:

- send a real number to add it;
- send `TotalCommand.SNAPSHOT` to obtain the current state without changing it;
- send `TotalCommand.RESET` to clear the state;
- send `TotalCommand.STOP` to terminate and return the final `TotalState`;
- reject booleans even though `bool` is a subclass of `int`;
- for unsupported messages, yield a response with `accepted=False` while preserving state.


### Solution 3


In [9]:
def running_total() -> Generator[TotalState, Real | TotalCommand | object, TotalState]:
    total = 0.0
    count = 0
    response = TotalState(total, count, note='ready')

    while True:
        message = yield response

        if message is TotalCommand.STOP:
            return TotalState(total, count, note='stopped')

        if message is TotalCommand.SNAPSHOT:
            response = TotalState(total, count, note='snapshot')
        elif message is TotalCommand.RESET:
            total = 0.0
            count = 0
            response = TotalState(total, count, note='reset')
        elif isinstance(message, Real) and not isinstance(message, bool):
            total += float(message)
            count += 1
            response = TotalState(total, count, note='value accepted')
        else:
            response = TotalState(
                total,
                count,
                accepted=False,
                note=f'unsupported message: {message!r}',
            )


In [10]:
totaler, initial = prime(running_total())
assert initial == TotalState(0.0, 0, note='ready')
assert totaler.send(10).total == 10
assert totaler.send(2.5) == TotalState(12.5, 2, note='value accepted')
assert totaler.send(True).accepted is False
assert totaler.send(TotalCommand.SNAPSHOT).count == 2
assert totaler.send(TotalCommand.RESET).total == 0

try:
    totaler.send(TotalCommand.STOP)
except StopIteration as exc:
    final_total_state = exc.value

assert final_total_state == TotalState(0.0, 0, note='stopped')


### Best practice

Define the inbound and outbound protocol as data types. It becomes much easier to document valid operations, reject invalid messages, and add commands later without ambiguous sentinel values.


---

# 4. A resizable rolling window

A stateful coroutine is useful when every incoming value updates a compact summary. The next problem adds dynamic configuration while the coroutine is running.


In [11]:
@dataclass(frozen=True)
class ResizeWindow:
    size: int


@dataclass(frozen=True)
class ResetWindow:
    pass


@dataclass(frozen=True)
class WindowStats:
    values: tuple[float, ...]
    size: int
    mean: float | None
    minimum: float | None
    maximum: float | None
    accepted: bool = True
    note: str = ''


## Problem 4 — Rolling statistics with live resizing

Implement `rolling_window(initial_size)`.

Requirements:

- accept real numbers and retain only the newest `size` values;
- accept `ResizeWindow(new_size)` while preserving the newest values that fit;
- accept `ResetWindow()` to clear all values;
- reject nonpositive sizes without changing the current configuration;
- yield a complete immutable snapshot after every message.


### Solution 4


In [12]:
def rolling_window(
    initial_size: int,
) -> Generator[WindowStats, Real | ResizeWindow | ResetWindow | object, None]:
    if initial_size <= 0:
        raise ValueError('initial_size must be positive')

    size = initial_size
    values: deque[float] = deque(maxlen=size)

    def snapshot(*, accepted: bool = True, note: str = '') -> WindowStats:
        frozen = tuple(values)
        return WindowStats(
            values=frozen,
            size=size,
            mean=(sum(frozen) / len(frozen)) if frozen else None,
            minimum=min(frozen) if frozen else None,
            maximum=max(frozen) if frozen else None,
            accepted=accepted,
            note=note,
        )

    response = snapshot(note='ready')

    while True:
        message = yield response

        if isinstance(message, ResizeWindow):
            if message.size <= 0:
                response = snapshot(accepted=False, note='size must be positive')
                continue
            size = message.size
            values = deque(tuple(values)[-size:], maxlen=size)
            response = snapshot(note='resized')
        elif isinstance(message, ResetWindow):
            values.clear()
            response = snapshot(note='reset')
        elif isinstance(message, Real) and not isinstance(message, bool):
            values.append(float(message))
            response = snapshot(note='value accepted')
        else:
            response = snapshot(accepted=False, note='unsupported message')


In [13]:
window, ready = prime(rolling_window(3))
assert ready.size == 3 and ready.values == ()
assert window.send(10).values == (10.0,)
assert window.send(20).mean == 15
assert window.send(30).values == (10.0, 20.0, 30.0)
assert window.send(40).values == (20.0, 30.0, 40.0)
assert window.send(ResizeWindow(2)).values == (30.0, 40.0)
assert window.send(ResizeWindow(0)).accepted is False
assert window.send(ResetWindow()).values == ()
window.close()


### Extension exercise

Add a `median` field without sorting on every update. One approach is to maintain two heaps. That turns this exercise into a useful study of streaming order statistics.


---

# 5. Validation without state corruption

A common bug is to mutate state before validating the complete message. The safe order is:

1. validate,
2. compute candidate state,
3. commit the update,
4. produce a response.


In [14]:
@dataclass(frozen=True)
class Observation:
    name: str
    value: float


@dataclass(frozen=True)
class RangeState:
    count: int
    minimum: float | None
    maximum: float | None
    last_name: str | None
    accepted: bool
    error: str | None = None


## Problem 5 — A robust range tracker

Build a coroutine that accepts `Observation` objects and tracks count, minimum, and maximum.

Validation rules:

- `name` must be a nonempty string after trimming;
- `value` must be a finite real number;
- invalid observations must not alter any state;
- the coroutine should never crash because of bad input; instead, it yields an error response.


### Solution 5


In [15]:
from math import isfinite


def range_tracker() -> Generator[RangeState, object, None]:
    count = 0
    minimum: float | None = None
    maximum: float | None = None
    last_name: str | None = None
    response = RangeState(0, None, None, None, True)

    while True:
        message = yield response

        if not isinstance(message, Observation):
            response = RangeState(
                count, minimum, maximum, last_name, False,
                'expected an Observation',
            )
            continue

        if not isinstance(message.name, str) or not message.name.strip():
            response = RangeState(
                count, minimum, maximum, last_name, False,
                'name must be nonempty',
            )
            continue

        if (
            not isinstance(message.value, Real)
            or isinstance(message.value, bool)
            or not isfinite(float(message.value))
        ):
            response = RangeState(
                count, minimum, maximum, last_name, False,
                'value must be a finite real number',
            )
            continue

        value = float(message.value)
        count += 1
        minimum = value if minimum is None else min(minimum, value)
        maximum = value if maximum is None else max(maximum, value)
        last_name = message.name.strip()
        response = RangeState(count, minimum, maximum, last_name, True)


In [16]:
tracker, _ = prime(range_tracker())
assert tracker.send(Observation('cpu', 0.7)).count == 1
assert tracker.send(Observation('memory', 0.4)).minimum == 0.4

before_bad = tracker.send(Observation('disk', 0.9))
after_bad = tracker.send(Observation('   ', 999))
assert after_bad.accepted is False
assert after_bad.count == before_bad.count
assert after_bad.maximum == before_bad.maximum

nan_response = tracker.send(Observation('network', float('nan')))
assert nan_response.accepted is False
assert nan_response.count == 3
tracker.close()


### Best practice

Treat every inbound message as untrusted. A coroutine is long-lived, so one malformed message should not silently poison all later results.


---

# 6. Injecting exceptions with `throw()`

`generator.throw(SomeError(...))` raises the supplied exception at the current suspension point. The generator may catch it and continue, or let it escape and close.

This creates a second control channel besides ordinary messages.


In [17]:
class SkipReading(Exception):
    """Discard the current logical reading and continue."""


class AbortReadings(Exception):
    """Stop processing and return a summary."""


@dataclass(frozen=True)
class ReadingSummary:
    accepted: tuple[float, ...]
    skipped: int


## Problem 6 — Recoverable and terminal exceptions

Implement `reading_collector()`.

- Sending a number appends it and yields the number of accepted values.
- Throwing `SkipReading` increments a skip counter and keeps the coroutine alive.
- Throwing `AbortReadings` returns a `ReadingSummary`.
- Any unrelated exception should propagate normally.


### Solution 6


In [18]:
def reading_collector() -> Generator[int, Real, ReadingSummary]:
    accepted: list[float] = []
    skipped = 0
    count = 0

    while True:
        try:
            value = yield count
        except SkipReading:
            skipped += 1
            continue
        except AbortReadings:
            return ReadingSummary(tuple(accepted), skipped)

        if not isinstance(value, Real) or isinstance(value, bool):
            raise TypeError('reading must be a real number')

        accepted.append(float(value))
        count += 1


In [19]:
collector, initial_count = prime(reading_collector())
assert initial_count == 0
assert collector.send(1.5) == 1
assert collector.throw(SkipReading('sensor timeout')) == 1
assert collector.send(2.5) == 2

try:
    collector.throw(AbortReadings('maintenance window'))
except StopIteration as exc:
    summary = exc.value

assert summary == ReadingSummary((1.5, 2.5), skipped=1)
assert getgeneratorstate(collector) == 'GEN_CLOSED'


### Important observation

A handled exception resumes the generator and may lead to another yielded value. An unhandled exception closes the generator. Therefore `throw()` belongs in protocols where exceptional control flow is explicitly documented.


---

# 7. Cleanup with `close()` and `finally`

Calling `close()` raises `GeneratorExit` at the suspension point. A generator should use `finally` for cleanup and should not yield another ordinary value while handling `GeneratorExit`.


## Problem 7 — A resource-safe buffered writer

Simulate a buffered writer coroutine.

- Each sent string is appended to an in-memory buffer.
- When the buffer reaches `batch_size`, flush it into `committed_batches`.
- On `close()`, flush any remaining records exactly once.
- Reject a nonpositive `batch_size` immediately.
- Return the number of pending records after each send.

We use lists instead of files so the behavior is deterministic and testable.


### Solution 7


In [20]:
def buffered_writer(
    committed_batches: list[tuple[str, ...]],
    batch_size: int,
) -> Generator[int, str, None]:
    if batch_size <= 0:
        raise ValueError('batch_size must be positive')

    buffer: list[str] = []

    def flush() -> None:
        if buffer:
            committed_batches.append(tuple(buffer))
            buffer.clear()

    try:
        pending = 0
        while True:
            record = yield pending
            if not isinstance(record, str):
                raise TypeError('record must be a string')
            buffer.append(record)
            if len(buffer) >= batch_size:
                flush()
            pending = len(buffer)
    finally:
        flush()


In [21]:
batches: list[tuple[str, ...]] = []
writer, pending = prime(buffered_writer(batches, batch_size=3))
assert pending == 0
assert writer.send('a') == 1
assert writer.send('b') == 2
assert writer.send('c') == 0
assert batches == [('a', 'b', 'c')]
assert writer.send('d') == 1
writer.close()
assert batches == [('a', 'b', 'c'), ('d',)]

# Closing an already closed generator is harmless.
writer.close()
assert batches == [('a', 'b', 'c'), ('d',)]


### Context-manager wrapper

A context manager makes cleanup automatic even if client code raises an exception.


In [22]:
@contextmanager
def managed_coroutine(
    generator: Generator[YieldT, SendT, ReturnT],
) -> Iterator[tuple[Generator[YieldT, SendT, ReturnT], YieldT]]:
    generator, first = prime(generator)
    try:
        yield generator, first
    finally:
        generator.close()

managed_batches: list[tuple[str, ...]] = []
with managed_coroutine(buffered_writer(managed_batches, 2)) as (managed, first):
    assert first == 0
    managed.send('x')
    managed.send('y')
    managed.send('z')

assert managed_batches == [('x', 'y'), ('z',)]


### Best practice

Place cleanup in `finally`, not after the main loop. Code after an infinite loop is not guaranteed to run when the coroutine is closed.


---

# 8. Returning a result from a generator

A generator's `return value` becomes `StopIteration(value)`. This is useful when the generator yields intermediate progress but returns one final summary.


In [23]:
@dataclass(frozen=True)
class FinishBatch:
    pass


@dataclass(frozen=True)
class BatchResult:
    count: int
    checksum: int
    items: tuple[int, ...]


## Problem 8 — Checksum batch collector

Write a coroutine that:

- accepts nonnegative integers;
- yields the running checksum modulo `256`;
- returns a `BatchResult` when sent `FinishBatch()`;
- rejects booleans and negative integers;
- exposes a helper `send_and_capture()` that reports either a yielded value or a returned value without raising `StopIteration` to the caller.


### Solution 8


In [24]:
def checksum_collector() -> Generator[int, int | FinishBatch, BatchResult]:
    items: list[int] = []
    checksum = 0

    while True:
        message = yield checksum
        if isinstance(message, FinishBatch):
            return BatchResult(len(items), checksum, tuple(items))
        if not isinstance(message, int) or isinstance(message, bool) or message < 0:
            raise ValueError('expected a nonnegative integer')
        items.append(message)
        checksum = (checksum + message) % 256


@dataclass(frozen=True)
class ResumeResult:
    yielded: Any = None
    returned: Any = None
    terminated: bool = False


def send_and_capture(generator: Generator[Any, Any, Any], value: Any) -> ResumeResult:
    try:
        return ResumeResult(yielded=generator.send(value))
    except StopIteration as exc:
        return ResumeResult(returned=exc.value, terminated=True)


In [25]:
checksum, first_checksum = prime(checksum_collector())
assert first_checksum == 0
assert send_and_capture(checksum, 100).yielded == 100
assert send_and_capture(checksum, 200).yielded == 44
result = send_and_capture(checksum, FinishBatch())
assert result.terminated is True
assert result.returned == BatchResult(2, 44, (100, 200))


### Best practice

If termination carries business data, wrap resumption in a helper that makes yielded and returned outcomes explicit. Do not use a broad `except Exception` to detect completion.


---

# 9. Delegating a two-way protocol with `yield from`

`yield from subgenerator` forwards all of the following automatically:

- values yielded by the subgenerator;
- values sent into the delegating generator;
- exceptions thrown into it;
- `close()` calls;
- the subgenerator's final return value.

This is much more than a loop over yielded values.


In [26]:
@dataclass(frozen=True)
class PhaseSummary:
    name: str
    count: int
    mean: float


@dataclass(frozen=True)
class PipelineSummary:
    phases: tuple[PhaseSummary, ...]


## Problem 9 — Multi-phase averaging via delegation

Create:

1. `average_phase(name, required_count)`, which receives exactly `required_count` numbers, yields the current mean after each one, and returns a `PhaseSummary`;
2. `multi_phase_average(specifications)`, which delegates to each phase with `yield from` and finally returns a `PipelineSummary`.

The client should interact with one generator object for the entire pipeline.


### Solution 9


In [27]:
def average_phase(
    name: str,
    required_count: int,
) -> Generator[float | None, Real, PhaseSummary]:
    if required_count <= 0:
        raise ValueError('required_count must be positive')

    total = 0.0
    count = 0
    mean: float | None = None

    while count < required_count:
        value = yield mean
        if not isinstance(value, Real) or isinstance(value, bool):
            raise TypeError('phase values must be real numbers')
        total += float(value)
        count += 1
        mean = total / count

    return PhaseSummary(name, count, total / count)


def multi_phase_average(
    specifications: Iterable[tuple[str, int]],
) -> Generator[float | None, Real, PipelineSummary]:
    summaries: list[PhaseSummary] = []
    for name, required_count in specifications:
        summary = yield from average_phase(name, required_count)
        summaries.append(summary)
    return PipelineSummary(tuple(summaries))


In [28]:
pipeline, first = prime(multi_phase_average([('warmup', 2), ('main', 3)]))
assert first is None
assert pipeline.send(10) == 10

# Completing warmup immediately enters the next delegated phase,
# whose first outward yield is None.
assert pipeline.send(20) is None
assert pipeline.send(4) == 4
assert pipeline.send(8) == 6

try:
    pipeline.send(12)
except StopIteration as exc:
    pipeline_summary = exc.value

assert pipeline_summary == PipelineSummary((
    PhaseSummary('warmup', 2, 15.0),
    PhaseSummary('main', 3, 8.0),
))


### Pause and reason

Why did sending `20` yield `None` instead of `15`?

The value `15` became the internal state just before the first phase returned. Once that subgenerator returned, `yield from` immediately started the next phase, and the next phase's initial `yield mean` produced `None`.

This transition behavior is subtle and is one reason protocols should document what is yielded at phase boundaries.


---

# 10. Coroutine as a state machine

A coroutine naturally stores state between messages. That makes it suitable for small parsers and protocol handlers.


In [29]:
@dataclass(frozen=True)
class ParserReply:
    state: str
    event: str
    record_name: str | None = None
    fields: tuple[str, ...] = ()
    error: str | None = None


## Problem 10 — Parse a tiny record protocol

Input lines follow this grammar:

```text
BEGIN <name>
<any number of field lines>
END
```

Create `record_parser()` that receives one line at a time and yields a `ParserReply`.

Rules:

- blank lines outside records are ignored;
- `BEGIN` inside an active record is an error but does not destroy the current record;
- `END` outside a record is an error;
- field lines outside a record are errors;
- a valid `END` emits a complete record and resets the parser;
- the parser remains alive after recoverable errors.


### Solution 10


In [30]:
def record_parser() -> Generator[ParserReply, str, None]:
    active_name: str | None = None
    fields: list[str] = []
    reply = ParserReply('idle', 'ready')

    while True:
        raw_line = yield reply

        if not isinstance(raw_line, str):
            reply = ParserReply(
                'recording' if active_name else 'idle',
                'error',
                active_name,
                tuple(fields),
                'line must be a string',
            )
            continue

        line = raw_line.strip()

        if not line and active_name is None:
            reply = ParserReply('idle', 'ignored-blank')
        elif line.startswith('BEGIN '):
            proposed_name = line[6:].strip()
            if active_name is not None:
                reply = ParserReply(
                    'recording', 'error', active_name, tuple(fields),
                    'nested BEGIN is not allowed',
                )
            elif not proposed_name:
                reply = ParserReply('idle', 'error', error='missing record name')
            else:
                active_name = proposed_name
                fields.clear()
                reply = ParserReply('recording', 'begin', active_name)
        elif line == 'END':
            if active_name is None:
                reply = ParserReply('idle', 'error', error='END without BEGIN')
            else:
                completed_name = active_name
                completed_fields = tuple(fields)
                active_name = None
                fields.clear()
                reply = ParserReply(
                    'idle', 'record', completed_name, completed_fields,
                )
        elif active_name is None:
            reply = ParserReply(
                'idle', 'error', error='field outside a record',
            )
        else:
            fields.append(line)
            reply = ParserReply(
                'recording', 'field', active_name, tuple(fields),
            )


In [31]:
parser, parser_ready = prime(record_parser())
assert parser_ready.event == 'ready'
assert parser.send('BEGIN user-42').event == 'begin'
assert parser.send('name=Ada').fields == ('name=Ada',)
assert parser.send('BEGIN nested').error == 'nested BEGIN is not allowed'
assert parser.send('role=engineer').fields == ('name=Ada', 'role=engineer')
completed = parser.send('END')
assert completed == ParserReply(
    'idle', 'record', 'user-42', ('name=Ada', 'role=engineer')
)
assert parser.send('orphan-field').error == 'field outside a record'
parser.close()


### Extension exercise

Add a `CANCEL` line that discards the current record and emits a cancellation event. Then add a strict mode in which any protocol error terminates the parser.


---

# 11. Fan-out and routing

A coroutine can act as a sink: it receives values but does not need to expose meaningful outward data. Multiple sinks can then be connected behind a dispatcher.


In [32]:
def append_sink(target: list[Any]) -> Generator[None, Any, None]:
    while True:
        target.append((yield None))


def sum_sink(target: list[float]) -> Generator[None, Real, None]:
    total = 0.0
    while True:
        value = yield None
        if not isinstance(value, Real) or isinstance(value, bool):
            raise TypeError('sum sink accepts real numbers')
        total += float(value)
        target[:] = [total]


## Problem 11 — Broadcast with failure isolation

Implement `broadcast(targets, on_error)`.

- Prime all target coroutines once.
- Every incoming message is sent to every still-active target.
- If a target raises an exception, call `on_error(index, exception)`, close that target, and remove it from future broadcasts.
- Yield the number of active targets after each message.
- Closing the broadcaster must close all remaining targets.


### Solution 11


In [33]:
def broadcast(
    targets: Iterable[Generator[Any, Any, Any]],
    on_error: Callable[[int, Exception], None],
) -> Generator[int, Any, None]:
    active: list[tuple[int, Generator[Any, Any, Any]]] = []

    for index, target in enumerate(targets):
        target, _ = prime(target)
        active.append((index, target))

    try:
        active_count = len(active)
        while True:
            message = yield active_count
            survivors: list[tuple[int, Generator[Any, Any, Any]]] = []

            for index, target in active:
                try:
                    target.send(message)
                except Exception as exc:
                    on_error(index, exc)
                    target.close()
                else:
                    survivors.append((index, target))

            active = survivors
            active_count = len(active)
    finally:
        for _, target in active:
            target.close()


In [34]:
all_messages: list[Any] = []
running_sum: list[float] = []
errors: list[tuple[int, str]] = []

hub, active_count = prime(broadcast(
    [append_sink(all_messages), sum_sink(running_sum)],
    lambda index, exc: errors.append((index, type(exc).__name__)),
))

assert active_count == 2
assert hub.send(3) == 2
assert hub.send(4) == 2
assert all_messages == [3, 4]
assert running_sum == [7.0]

# The append sink accepts this; the numeric sink fails and is isolated.
assert hub.send('not numeric') == 1
assert all_messages == [3, 4, 'not numeric']
assert errors == [(1, 'TypeError')]
assert hub.send(10) == 1
hub.close()


### Best practice

Decide explicitly whether downstream failure should fail the whole pipeline or only one branch. Silent exception swallowing is almost never the right policy; report and isolate failures deliberately.


---

# 12. Online statistics with Welford's algorithm

A streaming algorithm should avoid storing every historical value. Welford's method updates mean and variance in one pass with good numerical stability.


In [35]:
class StatsCommand(Enum):
    RESET = auto()
    STOP = auto()


@dataclass(frozen=True)
class OnlineStats:
    count: int
    mean: float | None
    variance: float | None
    standard_deviation: float | None


## Problem 12 — Streaming mean and sample variance

Implement `online_statistics()` using Welford's algorithm.

For each numeric value:

```text
count += 1
delta = value - mean
mean += delta / count
delta2 = value - mean
M2 += delta * delta2
```

The sample variance is `M2 / (count - 1)` when `count >= 2`.

Also support reset and stop commands. On stop, return the last `OnlineStats` snapshot.


### Solution 12


In [36]:
def online_statistics() -> Generator[OnlineStats, Real | StatsCommand, OnlineStats]:
    count = 0
    mean = 0.0
    m2 = 0.0

    def snapshot() -> OnlineStats:
        variance = m2 / (count - 1) if count >= 2 else None
        return OnlineStats(
            count=count,
            mean=mean if count else None,
            variance=variance,
            standard_deviation=sqrt(variance) if variance is not None else None,
        )

    response = snapshot()

    while True:
        message = yield response

        if message is StatsCommand.STOP:
            return snapshot()
        if message is StatsCommand.RESET:
            count = 0
            mean = 0.0
            m2 = 0.0
            response = snapshot()
            continue
        if not isinstance(message, Real) or isinstance(message, bool):
            raise TypeError('statistics input must be a real number')

        value = float(message)
        count += 1
        delta = value - mean
        mean += delta / count
        delta2 = value - mean
        m2 += delta * delta2
        response = snapshot()


In [37]:
stats, empty_stats = prime(online_statistics())
assert empty_stats == OnlineStats(0, None, None, None)

for value in [2, 4, 4, 4, 5, 5, 7, 9]:
    latest_stats = stats.send(value)

assert latest_stats.count == 8
assert isclose(latest_stats.mean, 5.0)
assert isclose(latest_stats.variance, 32 / 7)
assert isclose(latest_stats.standard_deviation, sqrt(32 / 7))

reset_stats = stats.send(StatsCommand.RESET)
assert reset_stats.count == 0

try:
    stats.send(StatsCommand.STOP)
except StopIteration as exc:
    stopped_stats = exc.value

assert stopped_stats.count == 0


### Why this is better than storing all values

Memory usage remains constant regardless of stream length. This matters for telemetry, logs, sensor data, and any process expected to run indefinitely.


---

# 13. Exponentially weighted anomaly detector

An exponentially weighted moving average gives more influence to recent values. A second exponentially weighted quantity can estimate recent variance.


In [38]:
@dataclass(frozen=True)
class ConfigureDetector:
    alpha: float | None = None
    threshold: float | None = None


@dataclass(frozen=True)
class ResetDetector:
    pass


@dataclass(frozen=True)
class Detection:
    value: float | None
    mean: float | None
    deviation: float | None
    z_score: float | None
    anomalous: bool
    alpha: float
    threshold: float
    accepted: bool = True
    note: str = ''


## Problem 13 — Reconfigurable EWMA detector

Implement `ewma_detector(alpha=0.2, threshold=3.0)`.

Protocol:

- a real number updates the exponentially weighted mean and variance;
- `ConfigureDetector` may update `alpha`, `threshold`, or both;
- valid `alpha` is in `(0, 1]`;
- valid `threshold` is positive;
- `ResetDetector()` clears learned statistics but keeps configuration;
- the response contains a z-score and anomaly flag;
- invalid configuration must be rejected atomically: change nothing if any supplied field is invalid.


### Solution 13


In [39]:
def ewma_detector(
    alpha: float = 0.2,
    threshold: float = 3.0,
) -> Generator[Detection, Real | ConfigureDetector | ResetDetector | object, None]:
    if not 0 < alpha <= 1:
        raise ValueError('alpha must be in (0, 1]')
    if threshold <= 0:
        raise ValueError('threshold must be positive')

    mean: float | None = None
    variance = 0.0
    response = Detection(None, None, None, None, False, alpha, threshold, note='ready')

    while True:
        message = yield response

        if isinstance(message, ConfigureDetector):
            candidate_alpha = alpha if message.alpha is None else message.alpha
            candidate_threshold = threshold if message.threshold is None else message.threshold

            if not 0 < candidate_alpha <= 1 or candidate_threshold <= 0:
                response = Detection(
                    None, mean, sqrt(variance) if mean is not None else None,
                    None, False, alpha, threshold,
                    accepted=False,
                    note='invalid configuration; no changes applied',
                )
                continue

            alpha = float(candidate_alpha)
            threshold = float(candidate_threshold)
            response = Detection(
                None, mean, sqrt(variance) if mean is not None else None,
                None, False, alpha, threshold, note='configured',
            )
            continue

        if isinstance(message, ResetDetector):
            mean = None
            variance = 0.0
            response = Detection(
                None, None, None, None, False, alpha, threshold, note='reset',
            )
            continue

        if not isinstance(message, Real) or isinstance(message, bool):
            response = Detection(
                None, mean, sqrt(variance) if mean is not None else None,
                None, False, alpha, threshold,
                accepted=False,
                note='expected a real number or command',
            )
            continue

        value = float(message)

        if mean is None:
            mean = value
            variance = 0.0
            z_score = 0.0
            anomalous = False
        else:
            previous_mean = mean
            mean = alpha * value + (1 - alpha) * previous_mean
            innovation = value - previous_mean
            variance = (1 - alpha) * (variance + alpha * innovation * innovation)
            deviation = sqrt(variance)
            z_score = (value - mean) / deviation if deviation > 0 else 0.0
            anomalous = abs(z_score) >= threshold

        response = Detection(
            value=value,
            mean=mean,
            deviation=sqrt(variance),
            z_score=z_score,
            anomalous=anomalous,
            alpha=alpha,
            threshold=threshold,
            note='value processed',
        )


In [40]:
detector, detector_ready = prime(ewma_detector(alpha=0.3, threshold=2.0))
assert detector_ready.note == 'ready'

for value in [10, 10.2, 9.8, 10.1, 9.9]:
    normal = detector.send(value)

old_alpha = normal.alpha
invalid_config = detector.send(ConfigureDetector(alpha=2.0, threshold=1.0))
assert invalid_config.accepted is False
assert invalid_config.alpha == old_alpha
assert invalid_config.threshold == 2.0

configured = detector.send(ConfigureDetector(threshold=1.0))
assert configured.threshold == 1.0
possible_anomaly = detector.send(30)
assert possible_anomaly.value == 30
assert possible_anomaly.z_score is not None

reset = detector.send(ResetDetector())
assert reset.mean is None
assert reset.threshold == 1.0
detector.close()


### Discussion

This detector is educational, not a universal anomaly model. Real systems must consider seasonality, drift, missing values, correlated measurements, and alert suppression. The coroutine pattern is still valuable because configuration and observations share one explicit protocol.


---

# 14. Capstone — Configurable telemetry engine

The capstone combines the notebook's main ideas:

- structured inbound commands;
- immutable outward snapshots;
- validation before mutation;
- dynamic rolling-window configuration;
- constant-memory online statistics;
- graceful termination with a return value;
- deterministic tests.


In [41]:
@dataclass(frozen=True)
class Metric:
    name: str
    value: float


@dataclass(frozen=True)
class SetTelemetryWindow:
    size: int


@dataclass(frozen=True)
class TelemetrySnapshotCommand:
    pass


@dataclass(frozen=True)
class ResetTelemetry:
    pass


@dataclass(frozen=True)
class StopTelemetry:
    pass


@dataclass(frozen=True)
class TelemetrySnapshot:
    processed: int
    rejected: int
    last_metric: str | None
    lifetime_mean: float | None
    lifetime_variance: float | None
    window_size: int
    window_values: tuple[float, ...]
    window_mean: float | None
    minimum: float | None
    maximum: float | None
    accepted: bool = True
    note: str = ''


## Capstone problem

Implement `telemetry_engine(window_size=5)`.

### Data messages

`Metric(name, value)` is valid when:

- `name` is a nonempty string after trimming;
- `value` is a finite real number and not a boolean.

A valid metric updates:

- lifetime count and Welford statistics;
- lifetime minimum and maximum;
- a rolling window containing the newest values;
- the last metric name.

### Commands

- `SetTelemetryWindow(size)` changes the window while preserving newest values;
- `TelemetrySnapshotCommand()` produces a snapshot without mutation;
- `ResetTelemetry()` clears all learned state but keeps the window size;
- `StopTelemetry()` returns the final snapshot and terminates.

### Error policy

Invalid messages do not terminate the engine. They increment `rejected`, preserve all analytical state, and yield `accepted=False`.


### Capstone solution


In [42]:
def telemetry_engine(
    window_size: int = 5,
) -> Generator[
    TelemetrySnapshot,
    Metric | SetTelemetryWindow | TelemetrySnapshotCommand | ResetTelemetry | StopTelemetry | object,
    TelemetrySnapshot,
]:
    if window_size <= 0:
        raise ValueError('window_size must be positive')

    processed = 0
    rejected = 0
    last_metric: str | None = None
    mean = 0.0
    m2 = 0.0
    minimum: float | None = None
    maximum: float | None = None
    window: deque[float] = deque(maxlen=window_size)

    def snapshot(*, accepted: bool = True, note: str = '') -> TelemetrySnapshot:
        values = tuple(window)
        lifetime_variance = m2 / (processed - 1) if processed >= 2 else None
        return TelemetrySnapshot(
            processed=processed,
            rejected=rejected,
            last_metric=last_metric,
            lifetime_mean=mean if processed else None,
            lifetime_variance=lifetime_variance,
            window_size=window_size,
            window_values=values,
            window_mean=(sum(values) / len(values)) if values else None,
            minimum=minimum,
            maximum=maximum,
            accepted=accepted,
            note=note,
        )

    response = snapshot(note='ready')

    while True:
        message = yield response

        if isinstance(message, StopTelemetry):
            return snapshot(note='stopped')

        if isinstance(message, TelemetrySnapshotCommand):
            response = snapshot(note='snapshot')
            continue

        if isinstance(message, ResetTelemetry):
            processed = 0
            rejected = 0
            last_metric = None
            mean = 0.0
            m2 = 0.0
            minimum = None
            maximum = None
            window.clear()
            response = snapshot(note='reset')
            continue

        if isinstance(message, SetTelemetryWindow):
            if message.size <= 0:
                rejected += 1
                response = snapshot(
                    accepted=False,
                    note='window size must be positive',
                )
                continue

            window_size = message.size
            window = deque(tuple(window)[-window_size:], maxlen=window_size)
            response = snapshot(note='window resized')
            continue

        valid_metric = (
            isinstance(message, Metric)
            and isinstance(message.name, str)
            and bool(message.name.strip())
            and isinstance(message.value, Real)
            and not isinstance(message.value, bool)
            and isfinite(float(message.value))
        )

        if not valid_metric:
            rejected += 1
            response = snapshot(
                accepted=False,
                note='invalid metric or unsupported command',
            )
            continue

        # All validation has succeeded; now commit the state transition.
        value = float(message.value)
        processed += 1
        delta = value - mean
        mean += delta / processed
        delta2 = value - mean
        m2 += delta * delta2
        minimum = value if minimum is None else min(minimum, value)
        maximum = value if maximum is None else max(maximum, value)
        last_metric = message.name.strip()
        window.append(value)
        response = snapshot(note='metric accepted')


### Capstone verification


In [43]:
engine, initial = prime(telemetry_engine(window_size=3))
assert initial.processed == 0
assert initial.window_size == 3

s1 = engine.send(Metric('cpu', 10))
s2 = engine.send(Metric('cpu', 20))
s3 = engine.send(Metric('memory', 30))
s4 = engine.send(Metric('disk', 40))

assert s4.processed == 4
assert s4.window_values == (20.0, 30.0, 40.0)
assert isclose(s4.lifetime_mean, 25.0)
assert isclose(s4.lifetime_variance, 500 / 3)
assert s4.minimum == 10
assert s4.maximum == 40
assert s4.last_metric == 'disk'

bad = engine.send(Metric('', 999))
assert bad.accepted is False
assert bad.rejected == 1
assert bad.processed == 4
assert bad.maximum == 40

resized = engine.send(SetTelemetryWindow(2))
assert resized.window_values == (30.0, 40.0)
assert resized.window_mean == 35

bad_resize = engine.send(SetTelemetryWindow(0))
assert bad_resize.accepted is False
assert bad_resize.rejected == 2
assert bad_resize.window_size == 2

snapshot_only = engine.send(TelemetrySnapshotCommand())
assert snapshot_only.note == 'snapshot'
assert snapshot_only.processed == 4

try:
    engine.send(StopTelemetry())
except StopIteration as exc:
    final_snapshot = exc.value

assert final_snapshot.note == 'stopped'
assert final_snapshot.processed == 4
assert final_snapshot.rejected == 2
assert getgeneratorstate(engine) == 'GEN_CLOSED'


## Capstone extension problems

Try these without changing the existing public response type unless necessary:

1. Add per-metric statistics, keyed by metric name.
2. Add `RemoveMetric(name)` to forget one metric's state.
3. Add a maximum accepted absolute value as a live configuration option.
4. Add checkpoint and restore commands using only immutable data.
5. Delegate each metric's calculations to a subgenerator with `yield from`.
6. Build a broadcaster that feeds the telemetry engine and a persistence sink.
7. Inject a `MaintenanceMode` exception with `throw()` and recover without losing state.
8. Wrap the engine in a context manager that always records a final snapshot.


---

# 15. Testing checklist for generator coroutines

A mature test suite should cover more than the happy path.

## Lifecycle

- newly created state;
- first yielded value after priming;
- suspended state after normal sends;
- closed state after return, unhandled exception, or `close()`.

## Protocol

- every valid message type;
- every command;
- invalid message types;
- boundary values;
- state preservation after rejection.

## Termination

- returned value through `StopIteration.value`;
- repeated `close()` calls;
- cleanup after a client-side exception;
- exceptions injected with `throw()`.

## Composition

- transition between delegated subgenerators;
- downstream failure policy;
- closure propagation through dispatchers and `yield from`.


# Final takeaways

1. `yield` is both an outward value and an inward suspension point.
2. A just-created generator must be primed before receiving a non-`None` value.
3. Each `send()` returns the next value yielded outward; it does not return the value just sent inward.
4. Long-lived coroutines benefit from explicit command and response types.
5. Validate before mutating state.
6. Use `throw()` only as a documented control channel.
7. Put cleanup in `finally` and make closure automatic at ownership boundaries.
8. Capture `StopIteration.value` when generator return data matters.
9. `yield from` delegates the complete generator protocol, not only iteration.
10. Test lifecycle, invalid inputs, termination, and composition—not only numeric results.
